# 出席管理・授業後アンケート v2 — Colab 共有用ノート

このノートを上から実行すると、Colab 上で5画面（入口 / 生徒用 / 校舎スタッフ用 / 講師・教室担当用 / 校舎QR読取タブレット）を表示できます。MTG・関係者レビュー用です。

**共有の流れ（要約）**
1. GitHub に最新を push
2. リポジトリを Public にする（最も簡単）／または閲覧者に権限付与
3. このノートを Colab で開く
4. 「ランタイム > すべてのセルを実行」
5. 「ファイル > ドライブにコピーを保存」→「共有」でリンクを発行し、相手に送る

注意: Colab 上の画面操作はブラウザ内の一時状態です。複数人で同じ状態をリアルタイム共有するものではありません（各自の画面内でのみ反映）。

## 共有までに必要な作業（チェックリスト）

### A. リポジトリ準備（PC側・1回）
- [ ] 変更を commit して GitHub に push（`git push`）
- [ ] リポジトリを **Public** にする: GitHub → Settings → 最下部 Danger Zone → Change repository visibility → Public
  - private のままにしたい場合は末尾「private のまま使う場合」セルを使用
- [ ] （任意）ローカル表示確認: `uv run python main.py --directory mock_app_v2 --port 8010` → http://localhost:8010

### B. Colab で開く
- [ ] https://colab.research.google.com を開く
- [ ] 「ファイル > ノートブックを開く > GitHub」で `YutzMame/tokushin_app_mock` を検索し `mock_app_v2/colab_share_v2.ipynb` を開く（または新規ノートに下のセットアップセルを貼付）
- [ ] 「ランタイム > すべてのセルを実行」

### C. 共有
- [ ] 「ファイル > ドライブにコピーを保存」
- [ ] 右上「共有」→「リンクを知っている全員」を **閲覧者**（必要なら コメント可）に設定
- [ ] リンクをコピーして相手へ送付

## セットアップ（最初に実行）

下のセルでリポジトリを取得し、表示ヘルパーを読み込みます。Public リポジトリならそのまま成功します。

In [ ]:
# === セットアップ: リポジトリ取得 + 表示ヘルパー読込（最初に実行）===
import importlib.util, os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/YutzMame/tokushin_app_mock.git"
REPO_DIR = "tokushin_app_mock"

def find_helper():
    for base in (".", REPO_DIR, f"/content/{REPO_DIR}"):
        p = Path(base) / "mock_app_v2" / "colab_preview_v2.py"
        if p.exists():
            return p
    return None

helper = find_helper()
if helper is None:
    print("リポジトリを取得します...")
    res = subprocess.run(["git", "clone", REPO_URL], capture_output=True, text=True)
    print(res.stdout or "", res.stderr or "")
    helper = find_helper()

if helper is None:
    raise SystemExit(
        "リポジトリを取得できませんでした。\n"
        "・private の場合: GitHub で Public にする（最も簡単）か、末尾のトークン方式セルを使ってください。\n"
        "・既にローカルにある場合: リポジトリのルートでこのノートを開いてください。"
    )

os.chdir(helper.resolve().parents[1])  # リポジトリのルートへ
spec = importlib.util.spec_from_file_location("colab_preview_v2", "mock_app_v2/colab_preview_v2.py")
preview_v2 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(preview_v2)
print("OK: 表示ヘルパーを読み込みました。下のセルを順に実行してください。")


## 入口（ハブ）

4導線・参照CSVの読込状態・最近の操作ログ。

In [ ]:
preview_v2.show_index_v2()

## 生徒用

ホーム / 出席 / アンケート / お知らせ（未読バッジ）/ カレンダー / 学習ログ / 設定。起動時に未読の重要変更をモーダル表示。

In [ ]:
preview_v2.show_student_app_v2()

## 校舎スタッフ用

ホーム（未対応事項・本日の講座カード→出席状況テーブル/アンケート/講座変更/配布物）、リアルタイム入室、マスタ・設定（閲覧＋「編集」切替・フィルタ・CSV出力、担当者/申込マスタ）、異常・声かけ（上部タブ・担当者割当）、データ入出力。

In [ ]:
preview_v2.show_staff_app_v2()

## 講師・教室担当用

講座一覧（本日のみ）、授業中管理、配布物一覧と欠席者受け渡し、アンケート傾向、次回メモ。

In [ ]:
preview_v2.show_teacher_app_v2()

## 校舎QR読取タブレット

管理者ログイン（admin / admin1234）→ 講座選択 → 生徒提示QR読取 → 登録/エラー表示（QR読取専用）。

In [ ]:
preview_v2.show_tablet_qr_app_v2()

## （private のまま使う場合）トークンでクローン

リポジトリを Public にしない場合のみ、下のセルでトークンを使ってクローンします。共有相手にもリポジトリ閲覧権限と各自のトークンが必要になるため、関係者だけで使う想定です。

In [ ]:
# === （private のまま使う場合のみ）Personal Access Token でクローン ===
# GitHub の Settings > Developer settings > Personal access tokens で repo 読取権限のトークンを発行。
# 実行するとトークン入力欄が出ます（ノートには保存されません）。実行後、上のセットアップセルを再実行。
import getpass, subprocess, os
token = getpass.getpass("GitHub Personal Access Token: ")
subprocess.run(["git", "clone", f"https://{token}@github.com/YutzMame/tokushin_app_mock.git"])
if os.path.isdir("tokushin_app_mock"):
    os.chdir("tokushin_app_mock")
print("クローン完了したら、上のセットアップセルをもう一度実行してください。")
